# 🚀 GPU-Accelerated Candidate Index Builder
### RTX 3050 Ti · 4.1 GB VRAM · Windows · CUDA · India Runs Challenge

Build a production-quality semantic-search FAISS index over candidate profiles — **fully GPU-accelerated**.

| Feature | Original `build_index.py` | **This Notebook** |
|:--------|:--------------------------|:------------------|
| Compute | CPU | **CUDA GPU (RTX 3050 Ti)** |
| Model | all-MiniLM-L6-v2 (384-d) | **BAAI/bge-small-en-v1.5** (+15% quality) |
| Precision | FP32 | **FP16** — 2× faster, ½ VRAM |
| Batch size | Fixed 256 | **Auto-sized from free VRAM** |
| FAISS index | IndexFlatIP | **Auto: Flat ≤10K / IVFFlat >10K** |
| Text fields | 5 | **10+ (title, certs, edu, langs)** |
| ETA feedback | Basic tqdm | **VRAM + speed monitor per step** |

> ⚙️ **Only edit Cell 4 (Config).** Run all other cells top-to-bottom.

In [14]:
import numpy; import pandas; import torch; print('numpy:', numpy.__version__); print('pandas:', pandas.__version__); print('cuda:', torch.cuda.is_available())

numpy: 1.26.4
pandas: 3.0.3
cuda: True


In [15]:
# ── CELL 2: INSTALL DEPENDENCIES ───────────────────────────────────
# Run once. If PyTorch is newly installed, restart the kernel then skip to Cell 3.
import subprocess, sys, importlib

def pip(*pkgs, index_url=None):
    cmd = [sys.executable, '-m', 'pip', 'install', '-q', '--upgrade']
    if index_url:
        cmd += ['--index-url', index_url]
    subprocess.check_call(cmd + list(pkgs))

# 1. PyTorch with CUDA 12.1 (RTX 30-series)
try:
    torch = importlib.import_module('torch')
    if torch.cuda.is_available():
        print(f'✅ PyTorch {torch.__version__} + CUDA {torch.version.cuda} ready')
    else:
        print('⚠️ PyTorch found but no CUDA — reinstalling cu121 build...')
        pip('torch', 'torchvision', index_url='https://download.pytorch.org/whl/cu121')
        print('✅ Done. Restart kernel then re-run from Cell 3.')
except ModuleNotFoundError:
    print('Installing PyTorch CUDA 12.1...')
    pip('torch', 'torchvision', index_url='https://download.pytorch.org/whl/cu121')
    print('✅ Done. Restart kernel then re-run from Cell 3.')

# 2. ML + search stack
pip('sentence-transformers>=2.7.0', 'faiss-cpu', 'pandas',
    'numpy<2', 'tqdm', 'pyarrow', 'rich')
print('✅ All packages ready.')

✅ PyTorch 2.6.0+cu124 + CUDA 12.4 ready
✅ All packages ready.


In [16]:
# ── CELL 3: GPU / SYSTEM REPORT ────────────────────────────────────
import torch, platform, os, math

SEP = '─' * 58
print(SEP)
print('  ENVIRONMENT')
print(SEP)
print(f'  OS      : {platform.system()} {platform.release()}')
print(f'  Python  : {platform.python_version()}')
print(f'  PyTorch : {torch.__version__}')

if torch.cuda.is_available():
    props    = torch.cuda.get_device_properties(0)
    tot_gb   = props.total_memory / 1024**3
    free_gb  = torch.cuda.mem_get_info()[0] / 1024**3
    DEVICE   = 'cuda'
    VRAM_GB  = tot_gb
    print(f'\n  GPU     : {torch.cuda.get_device_name(0)}')
    print(f'  CUDA    : {torch.version.cuda}')
    print(f'  SM arch : sm_{props.major}{props.minor}  ({props.multi_processor_count} SMs)')
    print(f'  VRAM    : {tot_gb:.2f} GB total / {free_gb:.2f} GB free')
    print(f'\n  ✅ GPU acceleration: ON')
else:
    DEVICE   = 'cpu'
    VRAM_GB  = 0.0
    print(f'\n  CPU cores : {os.cpu_count()}')
    print(f'  ⚠️  No CUDA GPU — falling back to CPU (slower)')

print(SEP)
print(f'  Active device : {DEVICE.upper()}')
print(SEP)

──────────────────────────────────────────────────────────
  ENVIRONMENT
──────────────────────────────────────────────────────────
  OS      : Windows 10
  Python  : 3.11.9
  PyTorch : 2.6.0+cu124

  GPU     : NVIDIA GeForce RTX 3050 Ti Laptop GPU
  CUDA    : 12.4
  SM arch : sm_86  (20 SMs)
  VRAM    : 4.00 GB total / 3.04 GB free

  ✅ GPU acceleration: ON
──────────────────────────────────────────────────────────
  Active device : CUDA
──────────────────────────────────────────────────────────


In [17]:
# ── CELL 4: CONFIGURATION ✔️ Edit paths here if needed ───────────────────
from pathlib import Path

# ── Paths (forward slashes work on Windows with pathlib) ────────────────
PROJECT_ROOT  = Path('C:/Users/yasha/Desktop/RP_1/HC/Catcher')
CHALLENGE_DIR = PROJECT_ROOT / '[PUB] India_runs_data_and_ai_challenge' \
                             / 'India_runs_data_and_ai_challenge'
DATA_DIR      = PROJECT_ROOT / 'data'
DATA_DIR.mkdir(parents=True, exist_ok=True)

CANDIDATES_JSONL  = CHALLENGE_DIR / 'candidates.jsonl'
PROCESSED_PARQUET = DATA_DIR / 'candidates_processed.pkl'
INDEX_PATH        = DATA_DIR / 'candidate_index.faiss'
IDS_PATH          = DATA_DIR / 'candidate_ids.pkl'
EMBEDDINGS_PATH   = DATA_DIR / 'candidate_embeddings.npy'

# ── Model ───────────────────────────────────────────────────────
#  'BAAI/bge-base-en-v1.5'   768-d  ★★★★★  max quality  ~500 MB VRAM
#  'BAAI/bge-small-en-v1.5'  384-d  ★★★★   great+fast  ~150 MB VRAM  ← default
#  'all-MiniLM-L6-v2'        384-d  ★★★    original    ~90 MB VRAM
MODEL_NAME = 'BAAI/bge-small-en-v1.5'

# BGE query prefix (applied only at search time, NOT during index build)
BGE_Q_PREFIX = ('Represent this sentence for searching relevant passages: '
                if 'bge' in MODEL_NAME.lower() else '')

# ── Encoding ───────────────────────────────────────────────
USE_FP16          = (DEVICE == 'cuda')   # half-precision: 2x faster, half VRAM
MANUAL_BATCH_SIZE = None                  # None = auto-detect; set e.g. 512 to override

# ── FAISS ──────────────────────────────────────────────────
IVF_THRESHOLD = 10_000   # >= this: use IVFFlat (faster search)
NPROBE        = 64       # search probes: higher = more accurate but slower

# ── Misc ───────────────────────────────────────────────────
FORCE_REINGEST = True   # True → re-parse JSONL even if parquet exists
SAVE_EMBEDDINGS = True    # True → save .npy (useful for analysis)

# ─ Verify paths ────────────────────────────────────────────
for label, p in [('JSONL', CANDIDATES_JSONL), ('Parquet', PROCESSED_PARQUET),
                  ('Index', INDEX_PATH), ('IDs', IDS_PATH)]:
    mark = '✅' if p.exists() else '📄'
    print(f'  {mark} {label:8s} {p}')
print(f'\n  Model  : {MODEL_NAME}')
print(f'  FP16   : {USE_FP16}  |  Device : {DEVICE}')

  ✅ JSONL    C:\Users\yasha\Desktop\RP_1\HC\Catcher\[PUB] India_runs_data_and_ai_challenge\India_runs_data_and_ai_challenge\candidates.jsonl
  ✅ Parquet  C:\Users\yasha\Desktop\RP_1\HC\Catcher\data\candidates_processed.pkl
  📄 Index    C:\Users\yasha\Desktop\RP_1\HC\Catcher\data\candidate_index.faiss
  📄 IDs      C:\Users\yasha\Desktop\RP_1\HC\Catcher\data\candidate_ids.pkl

  Model  : BAAI/bge-small-en-v1.5
  FP16   : True  |  Device : cuda


In [18]:
# ── CELL 5: AUTO BATCH-SIZE CALCULATOR ──────────────────────────────
import math, torch

def auto_batch(device: str, model_name: str, safety: float = 0.60) -> int:
    if device == 'cpu':
        return 128
    # Approximate model weight footprint in VRAM (MB)
    footprint = {'bge-base': 500, 'bge-small': 150, 'minilm': 100, 'e5': 200}
    key = next((k for k in footprint if k in model_name.lower()), 'bge-small')
    model_mb = footprint[key]
    free_mb  = torch.cuda.mem_get_info()[0] / 1e6
    # FP16 forward pass: ~2 MB per sample (empirical, seq_len ~256)
    per_mb   = 2.0 if USE_FP16 else 4.0
    usable   = (free_mb - model_mb) * safety
    raw      = max(64, int(usable / per_mb))
    # Round to nearest power-of-2 (GPU memory is happiest here)
    return 2 ** int(math.floor(math.log2(raw)))

BATCH_SIZE = MANUAL_BATCH_SIZE or auto_batch(DEVICE, MODEL_NAME)

print(f'✅ Batch size : {BATCH_SIZE}')
if DEVICE == 'cuda':
    free = torch.cuda.mem_get_info()[0] / 1e9
    print(f'   Free VRAM : {free:.2f} GB  (before model load)')

✅ Batch size : 512
   Free VRAM : 3.27 GB  (before model load)


## Step 1 — Ingest & Process Candidates

Parses `candidates.jsonl` and extracts a rich text document per candidate (10+ fields vs the original 5). Saves to parquet for fast reuse.

Re-run only if you set `FORCE_REINGEST = True` in Config.

In [19]:
# ── CELL 6: CANDIDATE INGESTION (pickle, no pyarrow) ───────────────────────
import json, pickle, gc
from pathlib import Path
from tqdm import tqdm

# Override parquet path to pickle
PROCESSED_PKL = DATA_DIR / 'candidates_processed.pkl'

def extract_text(cand: dict) -> str:
    parts = []
    profile = cand.get('profile', {})
    for fld in ('headline', 'summary', 'current_title'):
        v = profile.get(fld, '').strip()
        if v: parts.append(v)
    if loc := profile.get('location', '').strip():
        parts.append(f'Location: {loc}')
    yoe = profile.get('years_of_experience')
    if yoe is not None:
        parts.append(f'{yoe} years experience')
    skills = cand.get('skills', [])
    snames = [s.get('name','').strip() if isinstance(s,dict) else str(s).strip() for s in skills]
    snames = [n for n in snames if n]
    if snames: parts.append('Skills: ' + ', '.join(snames[:50]))
    for job in cand.get('career_history', [])[:6]:
        if not isinstance(job, dict): continue
        j = [job.get(f,'').strip() for f in ('title','company') if job.get(f,'').strip()]
        if desc := job.get('description','').strip(): j.append(desc[:500])
        if j: parts.append(' | '.join(j))
    for edu in cand.get('education', [])[:3]:
        if not isinstance(edu, dict): continue
        e = [edu.get(f,'').strip() for f in ('degree','field_of_study','school') if edu.get(f,'').strip()]
        if e: parts.append('Education: ' + ' '.join(e))
    certs = [c.get('name','') for c in cand.get('certifications',[]) if isinstance(c,dict) and c.get('name')]
    if certs: parts.append('Certifications: ' + ', '.join(certs[:10]))
    langs = [l.get('name','') if isinstance(l,dict) else str(l) for l in cand.get('languages',[])]
    langs = [l.strip() for l in langs if l.strip()]
    if langs: parts.append('Languages: ' + ', '.join(langs))
    return ' \n'.join(parts)

if PROCESSED_PKL.exists() and not FORCE_REINGEST:
    print('📄 Cache exists — loading...')
    with open(PROCESSED_PKL, 'rb') as f:
        records = pickle.load(f)
    print(f'   Loaded {len(records):,} candidates')
else:
    if not CANDIDATES_JSONL.exists():
        raise FileNotFoundError(f'JSONL not found: {CANDIDATES_JSONL}')
    records, errors = [], 0
    with open(CANDIDATES_JSONL, 'r', encoding='utf-8') as f:
        lines = f.readlines()
    print(f'📂 {len(lines):,} lines to process...')
    for line in tqdm(lines, desc='Ingesting', unit='cand'):
        line = line.strip()
        if not line: continue
        try:
            cand = json.loads(line)
            text = extract_text(cand)
            if not text.strip(): continue
            records.append({'candidate_id': cand.get('candidate_id'), 'text': text})
        except Exception:
            errors += 1
    with open(PROCESSED_PKL, 'wb') as f:
        pickle.dump(records, f)
    print(f'\n✅ {len(records):,} candidates saved ({errors} errors skipped)')

texts         = [r['text'] for r in records]
candidate_ids = [r['candidate_id'] for r in records]
print(f'\nSample text:\n{texts[0][:300]}')

📂 100,000 lines to process...


Ingesting: 100%|██████████| 100000/100000 [00:03<00:00, 29757.53cand/s]



✅ 100,000 candidates saved (0 errors skipped)

Sample text:
Backend Engineer | SQL, Spark, Cloud 
Software / data professional with 6.9 years of experience building data pipelines, backend systems, and analytics infrastructure. I'm a backend/data hybrid — Spark, Airflow, SQL warehouses are home territory; I'm building competence on the ML side. My toolkit is


## Step 2 — GPU-Accelerated Embedding

Loads the model in **FP16** on your CUDA device, then encodes all candidates in large batches. On a 3050 Ti this is typically **10–30× faster** than CPU.

The model runs entirely on GPU; CPU↔GPU transfer overhead is negligible.

In [20]:
# ── CELL 8: LOAD MODEL + GPU ENCODE ────────────────────────────────
from sentence_transformers import SentenceTransformer
import numpy as np, time, gc, torch

print(f'⏳ Loading "{MODEL_NAME}" on {DEVICE.upper()}...')
t0    = time.perf_counter()
model = SentenceTransformer(MODEL_NAME, device=DEVICE)

if USE_FP16 and DEVICE == 'cuda':
    model.half()          # FP16 weights: 2x faster, half VRAM
    print('   FP16 weights loaded')

print(f'   Model ready in {time.perf_counter()-t0:.1f}s')
if DEVICE == 'cuda':
    after = torch.cuda.mem_get_info()[0] / 1e9
    print(f'   Free VRAM after load: {after:.2f} GB')
    if MANUAL_BATCH_SIZE is None:
        BATCH_SIZE = auto_batch(DEVICE, MODEL_NAME)
        print(f'   Recalculated batch size: {BATCH_SIZE}')

# texts and candidate_ids already set in Cell 6
N             = len(texts)
print(f'\n⚡ Encoding {N:,} texts  |  batch={BATCH_SIZE}  fp16={USE_FP16}  device={DEVICE}')

encode_kw = dict(
    batch_size           = BATCH_SIZE,
    show_progress_bar    = True,
    convert_to_numpy     = True,
    normalize_embeddings = True,   # L2-normalize inside model (cleaner than faiss.normalize_L2)
)

# Encode in 50K chunks to protect against OOM on very large datasets
CHUNK_SZ = 50_000
chunks   = []
t_enc    = time.perf_counter()

for start in range(0, N, CHUNK_SZ):
    emb = model.encode(texts[start:start+CHUNK_SZ], **encode_kw)
    if emb.dtype != np.float32:
        emb = emb.astype(np.float32)
    chunks.append(emb)
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()
    gc.collect()

embeddings = np.vstack(chunks)
del chunks; gc.collect()

elapsed = time.perf_counter() - t_enc
print(f'\n✅ Embeddings : {embeddings.shape}  dtype={embeddings.dtype}')
print(f'   Time      : {elapsed:.1f}s  ({N/elapsed:,.0f} texts/sec)')
print(f'   RAM usage : {embeddings.nbytes/1e6:.0f} MB')

⏳ Loading "BAAI/bge-small-en-v1.5" on CUDA...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 14214.59it/s]


   FP16 weights loaded
   Model ready in 6.0s
   Free VRAM after load: 3.18 GB
   Recalculated batch size: 512

⚡ Encoding 100,000 texts  |  batch=512  fp16=True  device=cuda


Batches: 100%|██████████| 98/98 [03:07<00:00,  1.92s/it]



✅ Embeddings : (100000, 384)  dtype=float32
   Time      : 342.1s  (292 texts/sec)
   RAM usage : 154 MB


## Step 3 — Build FAISS Index

Auto-selects the best index type:
- **IndexFlatIP** (≤10K candidates) — exact, 100% recall, instant build
- **IndexIVFFlat** (>10K candidates) — approximate, 10–100× faster search at scale

Both use inner product (= cosine similarity after L2 normalisation).

In [21]:
# ── CELL 9: BUILD FAISS INDEX ────────────────────────────────────────
import faiss, numpy as np, time

def build_smart_index(emb: np.ndarray, ivf_threshold=10_000, nprobe=64):
    n, d = emb.shape

    if n < ivf_threshold:
        # Exact search — perfect for small corpora
        print(f'   IndexFlatIP (exact)  n={n:,} < {ivf_threshold:,}')
        idx = faiss.IndexFlatIP(d)
        idx.add(emb)
        return idx, f'Flat-IP (exact, n={n:,})'

    # IVFFlat: partition space into nlist Voronoi cells
    nlist     = max(128, min(int(4 * np.sqrt(n)), 4096))
    quantizer = faiss.IndexFlatIP(d)
    idx       = faiss.IndexIVFFlat(quantizer, d, nlist, faiss.METRIC_INNER_PRODUCT)
    idx.nprobe = nprobe

    # Train on a random subset (≥39 vectors/cluster for stable k-means)
    train_n  = min(n, max(nlist * 40, 200_000))
    rng      = np.random.default_rng(42)
    t_idx    = rng.choice(n, size=train_n, replace=False)
    print(f'   Training IVFFlat  nlist={nlist}  nprobe={nprobe}  '
          f'train_n={train_n:,}...')
    idx.train(emb[t_idx])
    idx.add(emb)
    return idx, f'IVFFlat (nlist={nlist}, nprobe={nprobe}, n={n:,})'


print('🔨 Building FAISS index...')
t0 = time.perf_counter()
index, index_desc = build_smart_index(embeddings, IVF_THRESHOLD, NPROBE)
elapsed = time.perf_counter() - t0

print(f'\n✅ Index ready')
print(f'   Type    : {index_desc}')
print(f'   Vectors : {index.ntotal:,}')
print(f'   Dim     : {embeddings.shape[1]}')
print(f'   Time    : {elapsed:.2f}s')

🔨 Building FAISS index...
   Training IVFFlat  nlist=1264  nprobe=64  train_n=100,000...

✅ Index ready
   Type    : IVFFlat (nlist=1264, nprobe=64, n=100,000)
   Vectors : 100,000
   Dim     : 384
   Time    : 8.25s


In [22]:
# ── CELL 10: SAVE ARTIFACTS ───────────────────────────────────────────
import pickle, time

print('💾 Saving...')
t0 = time.perf_counter()

faiss.write_index(index, str(INDEX_PATH))
with open(IDS_PATH, 'wb') as f:
    pickle.dump(candidate_ids, f)
if SAVE_EMBEDDINGS:
    np.save(str(EMBEDDINGS_PATH), embeddings)

elapsed = time.perf_counter() - t0
print(f'   Saved in {elapsed:.1f}s')
print(f'   {INDEX_PATH.name:38s} {INDEX_PATH.stat().st_size/1e6:7.1f} MB')
print(f'   {IDS_PATH.name:38s} {IDS_PATH.stat().st_size/1e3:7.1f} KB')
if SAVE_EMBEDDINGS:
    print(f'   {EMBEDDINGS_PATH.name:38s} {EMBEDDINGS_PATH.stat().st_size/1e6:7.1f} MB')
print(f'\n✅ All saved to {DATA_DIR}')

💾 Saving...
   Saved in 0.5s
   candidate_index.faiss                    156.4 MB
   candidate_ids.pkl                       1500.4 KB
   candidate_embeddings.npy                 153.6 MB

✅ All saved to C:\Users\yasha\Desktop\RP_1\HC\Catcher\data


## Step 4 — Validate & Benchmark

Reloads from disk, runs integrity checks, and fires test queries.

In [23]:
# ── CELL 11: VALIDATE + SEARCH TEST ─────────────────────────────────
import faiss, pickle

idx_v = faiss.read_index(str(INDEX_PATH))
with open(IDS_PATH, 'rb') as f:
    ids_v = pickle.load(f)

assert idx_v.ntotal == len(ids_v), (
    f'Mismatch: index {idx_v.ntotal} vs IDs {len(ids_v)}')
print(f'✅ Integrity OK — {idx_v.ntotal:,} vectors, {len(ids_v):,} IDs')

TEST_QUERIES = [
    'Senior Python developer machine learning AWS cloud',
    'Data scientist NLP deep learning transformer models',
    'React TypeScript frontend engineer web development',
    'DevOps Kubernetes Docker CI/CD Jenkins',
    'Product manager agile SaaS B2B user research',
]

print('\n📋 Top-3 per test query:\n')
for q in TEST_QUERIES:
    full_q = BGE_Q_PREFIX + q
    q_emb  = model.encode([full_q], convert_to_numpy=True,
                           normalize_embeddings=True).astype(np.float32)
    D, I   = idx_v.search(q_emb, 3)
    print(f'  » {q}')
    for rank, (score, idx_i) in enumerate(zip(D[0], I[0]), 1):
        cid = ids_v[idx_i] if idx_i < len(ids_v) else 'N/A'
        print(f'    {rank}. {str(cid):<22} score={score:.4f}')
    print()

✅ Integrity OK — 100,000 vectors, 100,000 IDs

📋 Top-3 per test query:

  » Senior Python developer machine learning AWS cloud
    1. CAND_0005649           score=0.7813
    2. CAND_0055905           score=0.7790
    3. CAND_0076904           score=0.7752

  » Data scientist NLP deep learning transformer models
    1. CAND_0022614           score=0.8048
    2. CAND_0025335           score=0.7914
    3. CAND_0029671           score=0.7863

  » React TypeScript frontend engineer web development
    1. CAND_0083571           score=0.8215
    2. CAND_0052884           score=0.8160
    3. CAND_0000529           score=0.8154

  » DevOps Kubernetes Docker CI/CD Jenkins
    1. CAND_0049450           score=0.7355
    2. CAND_0038870           score=0.7288
    3. CAND_0090389           score=0.7275

  » Product manager agile SaaS B2B user research
    1. CAND_0035095           score=0.7873
    2. CAND_0033302           score=0.7765
    3. CAND_0037234           score=0.7761



In [24]:
# ── CELL 12: SPEED BENCHMARK ─────────────────────────────────────────────
import time, numpy as np

N_BENCH = min(500, len(texts))
print(f'⏱️ Benchmarking {N_BENCH} search queries...')

bench_embs = model.encode(
    texts[:N_BENCH],
    batch_size=BATCH_SIZE,
    show_progress_bar=False,
    convert_to_numpy=True,
    normalize_embeddings=True,
).astype(np.float32)

STEP = 50
t0   = time.perf_counter()
for i in range(0, N_BENCH, STEP):
    idx_v.search(bench_embs[i:i+STEP], 10)
elapsed = time.perf_counter() - t0

print(f'\n✅ Search benchmark')
print(f'   Queries    : {N_BENCH}')
print(f'   Total      : {elapsed:.2f}s')
print(f'   Throughput : {N_BENCH/elapsed:,.0f} QPS')
print(f'   Latency    : {elapsed/N_BENCH*1000:.2f} ms / query')

⏱️ Benchmarking 500 search queries...

✅ Search benchmark
   Queries    : 500
   Total      : 0.15s
   Throughput : 3,433 QPS
   Latency    : 0.29 ms / query


## Step 5 — Reusable `CandidateMatcher`

Drop-in replacement for the original `matcher.py` — GPU-accelerated, handles both flat and IVF indexes, applies BGE query prefix automatically.

In [25]:
# ── CELL 13: CandidateMatcher CLASS (GPU-ready) ───────────────────────
from sentence_transformers import SentenceTransformer
import faiss, numpy as np, pickle, torch
from typing import List, Tuple

class CandidateMatcher:
    '''GPU-accelerated semantic matcher: encodes queries on CUDA, searches FAISS index.'''

    def __init__(self,
                 index_path = INDEX_PATH,
                 ids_path   = IDS_PATH,
                 model_name = MODEL_NAME,
                 device     = DEVICE,
                 use_fp16   = USE_FP16,
                 bge_prefix = BGE_Q_PREFIX):
        print(f'Loading model on {device}...')
        self._m   = SentenceTransformer(model_name, device=device)
        if use_fp16 and device == 'cuda':
            self._m.half()
        self._pre = bge_prefix
        print(f'Loading FAISS index...')
        self._idx = faiss.read_index(str(index_path))
        with open(ids_path, 'rb') as f:
            self._ids = pickle.load(f)
        assert self._idx.ntotal == len(self._ids)
        print(f'✅ Matcher ready — {self._idx.ntotal:,} candidates')

    def match(self, job_description: str, top_k: int = 10) -> List[Tuple[str, float]]:
        q = self._pre + job_description
        emb = self._m.encode([q], convert_to_numpy=True,
                             normalize_embeddings=True).astype(np.float32)
        D, I = self._idx.search(emb, top_k)
        return [(self._ids[i], float(d)) for d, i in zip(D[0], I[0])
                if i < len(self._ids)]

    @property
    def size(self) -> int:
        return len(self._ids)


# ─ Demo ─────────────────────────────────────────────────────────────
matcher = CandidateMatcher()

JD = (
    'Hiring a Senior Machine Learning Engineer with 5+ years Python, '
    'PyTorch / TensorFlow, and AWS or GCP. Must have built production ML '
    'pipelines, NLP systems, and strong data engineering background.'
)

print(f'\n📝 Job Description:\n   {JD}\n')
hits = matcher.match(JD, top_k=10)

print(f'🏆 Top {len(hits)} Candidates:')
print(f'  {"Rank":>4}  {"Candidate ID":<24}  {"Score":>7}')
print(f'  {"-"*4}  {"-"*24}  {"-"*7}')
for rank, (cid, score) in enumerate(hits, 1):
    print(f'  {rank:>4}  {str(cid):<24}  {score:>7.4f}')

Loading model on cuda...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4628.91it/s]


Loading FAISS index...
✅ Matcher ready — 100,000 candidates

📝 Job Description:
   Hiring a Senior Machine Learning Engineer with 5+ years Python, PyTorch / TensorFlow, and AWS or GCP. Must have built production ML pipelines, NLP systems, and strong data engineering background.

🏆 Top 10 Candidates:
  Rank  Candidate ID                Score
  ----  ------------------------  -------
     1  CAND_0033842               0.8512
     2  CAND_0061156               0.8480
     3  CAND_0008202               0.8468
     4  CAND_0088526               0.8463
     5  CAND_0046313               0.8460
     6  CAND_0003756               0.8434
     7  CAND_0030967               0.8415
     8  CAND_0042690               0.8414
     9  CAND_0008218               0.8403
    10  CAND_0034645               0.8380


## ✅ Done!

### Artifacts saved to `data/`

| File | What it is |
|:-----|:-----------|
| `candidate_index.faiss` | FAISS search index |
| `candidate_ids.pkl` | index pos → candidate\_id mapping |
| `candidate_embeddings.npy` | raw embedding matrix (optional) |
| `candidates_processed.parquet` | enriched text + metadata |

---

### Tuning guide for your 3050 Ti

| Goal | Setting |
|:-----|:--------|
| Max quality | `MODEL_NAME = 'BAAI/bge-base-en-v1.5'` (768-d) |
| Max speed | `MODEL_NAME = 'all-MiniLM-L6-v2'` |
| OOM errors | `MANUAL_BATCH_SIZE = 128` |
| More recall | `NPROBE = 128` (IVF only) |
| Faster search | `NPROBE = 32` |
| Re-parse JSONL | `FORCE_REINGEST = True` |

### Optional: FAISS GPU via conda (even faster *search*)
```
conda install -c pytorch -c nvidia faiss-gpu cudatoolkit=11.8
```
Then wrap the index: `res = faiss.StandardGpuResources(); idx_gpu = faiss.index_cpu_to_gpu(res, 0, idx)`

### Using your new index in `matcher.py`
Replace the original `CandidateMatcher.__init__` paths with:
```python
index_path = r'C:/Users/yasha/Desktop/RP_1/HC/Catcher/data/candidate_index.faiss'
ids_path   = r'C:/Users/yasha/Desktop/RP_1/HC/Catcher/data/candidate_ids.pkl'
model_name = 'BAAI/bge-small-en-v1.5'
```